In [ ]:
#!/usr/bin/env python3
"""
src/plot_results.py
Generates publication-grade PDF & 300-DPI PNG figures for Section V (Results)
directly from the raw experimental CSV outputs.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 8.5,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})


def generate_all_result_figures(raw_dir: str = "results/raw_runs", data_dir: str = "data", fig_dir: str = "results/figures") -> None:
    os.makedirs(fig_dir, exist_ok=True)

    # 1. Figure 4: D1 Multi-Tier Predictive F1 Boxplots
    d1_t1 = pd.read_csv(os.path.join(raw_dir, "d1_predictive_Tier1_FeatureIDE.csv"))
    d1_t2 = pd.read_csv(os.path.join(raw_dir, "d1_predictive_Tier2_LinuxKernel.csv"))
    models = ["Static_Threshold", "PIM_Siegmund", "MLP_Neural", "Plain_GBDT", "Proposed_Hybrid_ML"]
    labels = ["Static\nThreshold", "PIM\n(Siegmund)", "MLP\nNeural", "Plain\nGBDT", "Proposed\nHybrid ML"]

    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.5), dpi=300, sharey=True)
    for ax, df_sub, title in [
        (axes[0], d1_t1, "(a) Tier 1: FeatureIDE (100 Features)"),
        (axes[1], d1_t2, "(b) Tier 2: Linux Kernel Kconfig (89 Features)"),
    ]:
        data_groups = [df_sub[df_sub["model"] == m]["f1"].values for m in models]
        bp = ax.boxplot(data_groups, patch_artist=True, labels=labels, widths=0.52)
        colors = ["#CBD5E1", "#93C5FD", "#6EE7B7", "#FCD34D", "#F97316"]
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.88)
        ax.set_title(title, fontsize=9, fontweight="bold")
        ax.grid(True, axis="y", linestyle="--", alpha=0.35)
        ax.tick_params(axis="x", labelsize=7.5)
    axes[0].set_ylabel("F1-Score (30 Stochastic Runs)", fontsize=8.5)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "fig4_d1_predictive_boxplots.pdf"), bbox_inches="tight")
    plt.close()

    # 2. Figure 5: D2 & D4 Closed-Loop Ablation (Sample Efficiency & Regret)
    d4_t1 = pd.read_csv(os.path.join(raw_dir, "d4_ablation_Tier1_FeatureIDE.csv"))
    variants = ["V1_OpenLoop_ML", "V2_Standalone_RL", "V3_ML_RL_NoXAI", "V4_Full_ClosedLoop"]
    v_labels = ["V1: Open-Loop\nML Only", "V2: Standalone\nSAT-RL", "V3: ML + RL\n(No XAI Mask)", "V4: Full Closed\nML-RL-XAI"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 3.4), dpi=300)
    means_ep = [d4_t1[d4_t1["variant"] == v]["episodes_to_95pct"].mean() for v in variants]
    stds_ep = [d4_t1[d4_t1["variant"] == v]["episodes_to_95pct"].std() for v in variants]
    ax1.bar(v_labels, means_ep, yerr=stds_ep, capsize=4, color=["#94A3B8", "#60A5FA", "#34D399", "#EA580C"], alpha=0.9)
    ax1.set_ylabel("Episodes to 95% Optimal Reward", fontsize=8.5)
    ax1.set_title("(a) Convergence Sample Efficiency (Lower is Better)", fontsize=9, fontweight="bold")
    ax1.tick_params(axis="x", labelsize=7.3)
    ax1.grid(True, axis="y", linestyle="--", alpha=0.35)

    means_reg = [d4_t1[d4_t1["variant"] == v]["nfp_regret"].mean() for v in variants]
    stds_reg = [d4_t1[d4_t1["variant"] == v]["nfp_regret"].std() for v in variants]
    ax2.bar(v_labels, means_reg, yerr=stds_reg, capsize=4, color=["#94A3B8", "#60A5FA", "#34D399", "#EA580C"], alpha=0.9)
    ax2.set_ylabel("Cumulative NFP Regret", fontsize=8.5)
    ax2.set_title("(b) Dynamic Perturbation Regret (Lower is Better)", fontsize=9, fontweight="bold")
    ax2.tick_params(axis="x", labelsize=7.3)
    ax2.grid(True, axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "fig5_d2_d4_ablation_curves.pdf"), bbox_inches="tight")
    plt.close()

    # 3. Figure 6: D3 Practitioner Study (Diagnostic Accuracy & Trust)
    df_user = pd.read_csv(os.path.join(data_dir, "practitioner_study_worksheet.csv"))
    c1 = df_user[df_user["condition"] == "C1_No_XAI"]
    c2 = df_user[df_user["condition"] == "C2_With_XAI"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.3), dpi=300)
    ax1.bar(["C1: Raw Output", "C2: Closed-Loop XAI"],
            [c1["diagnostic_accuracy"].mean() * 100, c2["diagnostic_accuracy"].mean() * 100],
            width=0.45, color=["#64748B", "#2F855A"])
    ax1.set_ylim(0, 105)
    ax1.set_ylabel("Root-Cause Diagnostic Accuracy (%)", fontsize=8.5)
    ax1.set_title("(a) Practitioner Diagnostic Accuracy (N=12)", fontsize=9, fontweight="bold")
    ax1.grid(True, axis="y", linestyle="--", alpha=0.35)

    ax2.boxplot([c1["likert_trust"].values, c2["likert_trust"].values],
                labels=["C1: Raw Output", "C2: Closed-Loop XAI"], patch_artist=True, widths=0.4)
    ax2.set_ylim(0.5, 5.5)
    ax2.set_ylabel("Stakeholder Trust Rating (1–5 Likert)", fontsize=8.5)
    ax2.set_title("(b) Perceived Trust & Auditability Score", fontsize=9, fontweight="bold")
    ax2.grid(True, axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "fig6_d3_practitioner_utility.pdf"), bbox_inches="tight")
    plt.close()

    # 4. Figure 7: D5 Sundermann Scalability (20 to 2,050 features)
    df_d5 = pd.read_csv(os.path.join(raw_dir, "d5_scalability_sundermann.csv"))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 3.4), dpi=300)
    ax1.scatter(df_d5["n_features"], df_d5["sat_rl_infer_ms"], color="#2B6CB0", alpha=0.8, label="SAT-Masked RL Inference")
    ax1.scatter(df_d5["n_features"], df_d5["shap_latency_ms"], color="#C05621", alpha=0.8, marker="^", label="TreeSHAP Attribution")
    ax1.set_xscale("log")
    ax1.set_xlabel("Number of Features (Log Scale, Tier 3 Sundermann)", fontsize=8.5)
    ax1.set_ylabel("Per-Decision Latency (ms)", fontsize=8.5)
    ax1.set_title("(a) Runtime Inference & XAI Latency Scaling", fontsize=9, fontweight="bold")
    ax1.grid(True, which="both", linestyle="--", alpha=0.35)
    ax1.legend(fontsize=7.5)

    ax2.scatter(df_d5["n_features"], df_d5["peak_ram_mb"], color="#2F855A", alpha=0.85)
    ax2.set_xscale("log")
    ax2.set_xlabel("Number of Features (Log Scale, Tier 3 Sundermann)", fontsize=8.5)
    ax2.set_ylabel("Peak Memory Footprint (MB)", fontsize=8.5)
    ax2.set_title("(b) Memory Overhead Across 50 Feature Models", fontsize=9, fontweight="bold")
    ax2.grid(True, which="both", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "fig7_d5_scalability.pdf"), bbox_inches="tight")
    plt.close()
    print(f"Generated all Section V figures in {fig_dir}/")